# Forecasting
Develop a **forecasting model** on the data stored in **your** InfluxDB: for example the zone temperature one hour ahead, or the energy of tomorrow.
A forecast is useful to a controller (pre-heat before it gets cold, shift consumption), which is why the last optional step publishes it on MQTT.

Steps: choose the target and the features, build lagged features, split train/test **by days**, beat a naive baseline, measure, plot.

In [ ]:
import base64, json, os, urllib.parse, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

INFLUX_URL = os.getenv("INFLUX_URL", "http://influxdb:8086")
INFLUX_USER = os.getenv("INFLUX_USERNAME", "")
INFLUX_PASS = os.getenv("INFLUX_PASSWORD", "")

def query(db, influxql):
    """Run one InfluxQL query on database `db` (InfluxDB 1.x HTTP API) and return a DataFrame.
    The index is the UTC time; the tags (if the query returns them) and the fields are columns.
    Use e.g.  SELECT * FROM points WHERE ...   or   ... GROUP BY point  (the group tags become columns)."""
    url = INFLUX_URL.rstrip("/") + "/query?" + urllib.parse.urlencode({"db": db, "q": influxql, "epoch": "ms"})
    req = urllib.request.Request(url)
    if INFLUX_USER:
        token = base64.b64encode(f"{INFLUX_USER}:{INFLUX_PASS}".encode()).decode()
        req.add_header("Authorization", "Basic " + token)
    with urllib.request.urlopen(req, timeout=60) as r:
        res = json.load(r)["results"][0]
    if "error" in res:
        raise RuntimeError(res["error"])
    frames = []
    for s in res.get("series", []):
        df = pd.DataFrame(s["values"], columns=s["columns"])
        for k, v in (s.get("tags") or {}).items():
            df[k] = v
        frames.append(df)
    if not frames:
        return pd.DataFrame()
    df = pd.concat(frames, ignore_index=True)
    if "time" not in df.columns:          # SHOW ... queries have no time
        return df
    df["time"] = pd.to_datetime(df["time"], unit="ms")
    return df.set_index("time").sort_index()

# Check the connection and look at what your subscriber wrote
print(query("base_building", "SHOW MEASUREMENTS"))

## 1. Look at your data
The databases are `base_building` (baseline run) and `controlled_building` (run with your controller).
The schema is YOURS (written by your `iot/subscriber.py`): list the measurements, tags and fields before choosing anything.

Note that a gateway with a deadband only writes a value when it changed: the series are **irregular**, so you must resample them to a regular time grid
(hold the last known value, `resample(...).last().ffill()`) before building lags.

In [ ]:
# TODO: explore your schema: SHOW MEASUREMENTS, SHOW TAG KEYS, SHOW FIELD KEYS, SHOW TAG VALUES ... WITH KEY = ...
# TODO: load the raw data into a DataFrame (query(db, "SELECT ... FROM ... WHERE ...")) and plot it

## 2. Target and features
TODO: decide what you predict and from what, e.g. target = zone temperature at t + 1 h; features = temperature now and in the past hours, electricity, hour of the day (sin/cos).
Only use what a controller could know **at time t**.

In [ ]:
STEP = "1h"          # TODO: resampling step
HORIZON = 1          # TODO: how many steps ahead
TARGET = None        # TODO: your series (column) to predict

# TODO: one column per series, regular time grid, last value held
# wide = raw.pivot_table(index=raw.index, columns="<tag with the series name>", values="<field>", aggfunc="last")
# wide = wide.resample(STEP).last().ffill().dropna()

In [ ]:
# TODO: lagged features (shift) and the target (shifted the other way). Drop the rows with NaN.
# X = pd.DataFrame(index=wide.index)
# for lag in range(0, 4): X[f"{TARGET}_lag{lag}"] = wide[TARGET].shift(lag)
# X["hour_sin"], X["hour_cos"] = ...
# y = wide[TARGET].shift(-HORIZON)

## 3. Split train / test **by days**
Not randomly: neighbouring hours are almost identical, a random split would leak the test data into the training and give an optimistic score.
Train on the first days, test on the last ones. TODO: choose the cut and say how many days each set has.

In [ ]:
# TODO: split by calendar day (X.index.normalize() / X.index.date)

## 4. Baseline: persistence
The naive forecast is "the value in one hour is the value now". A model is worth something only if it beats it on the test days.

In [ ]:
# TODO: persistence forecast on the test set and its MAE and RMSE

## 5. Your model
TODO: fit one model on the training days, e.g. `sklearn.linear_model.Ridge` / `LinearRegression` or `sklearn.ensemble.GradientBoostingRegressor`
(scikit-learn is installed; `lightgbm`/`xgboost` are not). Tip: predicting the **change** (y - value now) instead of y often works better.

In [ ]:
# from sklearn.linear_model import Ridge
# from sklearn.ensemble import GradientBoostingRegressor
# TODO: fit, predict the test set

## 6. Metrics and plot

In [ ]:
# TODO: MAE and RMSE of the model vs persistence on the test days (sklearn.metrics.mean_absolute_error, mean_squared_error)
# TODO: plot actual vs persistence vs model on the test days

## 7. Optional: serve the forecast to your controller
A controller that knows the next hour can act earlier. Turn the model into a small service `ai/forecaster.py`:

* every step it reads the latest values (from MQTT topics of your gateway, or from InfluxDB with `query`),
* builds the same features as above, predicts,
* publishes the forecast on an MQTT topic **of your own** (not starting with `platform/`), e.g. `forecast/<point>/1h`, as JSON, and
* `ai/controller.py` subscribes to that topic and uses the value in `decide()`.

Save the trained model with `joblib.dump(model, "ai/models/forecast.joblib")` and load it in the service.